<a href="https://colab.research.google.com/github/paras9o9/AI-powered-journaling-app/blob/main/notebooks/AI%E2%80%91powered_journaling_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **4-Class Journal Classifier**

- Setup and data loading:
  - Import libraries, set random seeds.
  - Load the label dataset.
  - Show label count as a sanity check.
- Label mapping and preprocessing:
  - Map {"NEU": 0, "HUMOR": 1, "MH": 2, "SI": 3} and store it as label_id.
  - Basic text cleaning/tokenisation as needed for TF-IDF.
- Train/validation/test split
  - Stratified 70/15/15 split on label_id to preserve SI and NEU proportions.
  - Confirm label distrbution in each split.
- Baseline models:
  - TF-IDF + LinearSVC + Explainable AI
  - TF-IDF + Logistic Regression + Explainable AI
- Main model:
  - DistilBERT 4-way classifier, with class weights or focal loss if needed becuase NEU is smaller + Explainable AI.
- Metrics:
  - Overall accuracy and macro F1.
  - Per-class precision/recall/F1.
  - Pay special attention to SI recall and SI false negative, since they drive safety logic.


In [ ]:
# # Cell 1 — Run FIRST, then Runtime → Restart Session
# !pip install --upgrade --force-reinstall \
#     torch==2.7.0+cu128 \
#     torchvision==0.22.0+cu128 \
#     torchaudio==2.7.0+cu128 \
#     --index-url https://download.pytorch.org/whl/cu128 \
#     --quiet
# !pip install "fsspec[http]==2025.3.0" --quiet

In [ ]:
# !pip uninstall -y numpy scipy scikit-learn
# !pip install -U numpy scipy scikit-learn

In [ ]:
# import os, sys
# os.kill(os.getpid(), 9)

# **Setup and data loading**


In [ ]:
import pandas as pd

LABEL2ID = {
    'NEU': 0,
    'HUMOR': 1,
    'MH': 2,
    'SI': 3
}

ID2LABEL = {v: k for k, v in LABEL2ID.items()}

def load_and_prepare_data(path: str) -> pd.DataFrame:

  df = pd.read_csv(path)

  df.columns = [c.strip().lower() for c in df.columns]
  if "combined_text" not in df.columns or "prelim_label" not in df.columns:
    raise ValueError(f"Exprected columns 'combined_text' and 'prelim_label', got: {df.columns.tolist()}" )

  df = df.dropna(subset=["combined_text", "prelim_label"])
  df = df[df['combined_text'].str.strip() != ""]

  df["prelim_label"] = (
      df["prelim_label"]
      .astype(str)
      .str.strip()
      .str.replace("^'+|'+S", "", regex=True)
      .str.upper()
  )

  valid_labels = set(LABEL2ID.keys())
  df = df[df["prelim_label"].isin(valid_labels)]

  df["label_id"] = df["prelim_label"].map(LABEL2ID)

  df = df.reset_index(drop=True)

  return df

In [ ]:
df = load_and_prepare_data("/content/SID_CLEANED_DATASET.csv")
print(df.head())
print(df["prelim_label"].value_counts())
print(df["label_id"].value_counts())

# **Train/Val/Test split**

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

def train_val_test_split(data: pd.DataFrame, seed: int = 42):
  train_df, temp_df = train_test_split(
      data,
      test_size=0.30,
      stratify=data["label_id"],
      random_state=seed
  )

  val_df, test_df = train_test_split(
      temp_df,
      test_size=0.50,
      stratify=temp_df["label_id"],
      random_state=seed
  )

  train_df = train_df.reset_index(drop=True)
  val_df = val_df.reset_index(drop=True)
  test_df = test_df.reset_index(drop=True)

  return train_df, val_df, test_df

In [ ]:
train_df, val_df, test_df = train_val_test_split(df, seed=42)

print("Train size:", len(train_df))
print("Val size:", len(val_df))
print("Test size:", len(test_df))

print("\nTrain label distribution")
print(train_df["label_id"].value_counts(normalize=True))

print("\nVal label distribution")
print(val_df["label_id"].value_counts(normalize=True))

print("\nTest label distribution")
print(test_df["label_id"].value_counts(normalize=True))

# **Visualisation**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.countplot(data=df, x='prelim_label')
plt.title('Distribution of Labels')
plt.xlabel('prelim_label')
plt.ylabel('Count')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(data=df, x='label_id')
plt.title('Distribution of Labels Ids')
plt.xlabel('label_id')
plt.ylabel('Count')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
def plot_label_distribution_splits(train_df, val_df, test_df, id2label):
  for df in [train_df, val_df, test_df]:
    df["label_name"] = df["label_id"].map(id2label)

  fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

  sns.countplot(data=train_df, x="label_name", ax=axes[0])
  axes[0].set_title("Train label distribution")
  axes[0].set_xlabel("Label")
  axes[0].set_ylabel("Count")

  sns.countplot(data=val_df, x="label_name", ax=axes[1])
  axes[1].set_title("Validation label distribution")
  axes[1].set_xlabel("Label")
  axes[1].set_ylabel("")

  sns.countplot(data=test_df, x="label_name", ax=axes[2])
  axes[2].set_title("Test label distribution")
  axes[2].set_xlabel("Label")
  axes[2].set_ylabel("")

  plt.tight_layout()
  plt.show()

plot_label_distribution_splits(train_df, val_df, test_df, ID2LABEL)

In [ ]:
def plot_label_proportions(train_df, val_df, test_df, id2label):
    def get_props(df, split_name):
        counts = df["label_id"].value_counts(normalize=True).rename(split_name)
        return counts

    train_props = get_props(train_df, "train")
    val_props = get_props(val_df, "val")
    test_props = get_props(test_df, "test")

    prop_df = pd.concat([train_props, val_props, test_props], axis=1)
    prop_df.index = prop_df.index.map(id2label)

    prop_df.plot(kind="bar", figsize=(8, 5))
    plt.ylabel("Proportion")
    plt.title("Label proportions across splits")
    plt.xticks(rotation=0)
    plt.ylim(0, 0.6)
    plt.show()

plot_label_proportions(train_df, val_df, test_df, ID2LABEL)

# **Model Development**
# TF-IDF + LinearSVC

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import os

def train_baseline_tfidf_model(train_df, val_df, id2label, save_path=None):
  clf_pipeline = Pipeline([
      ("tfidf", TfidfVectorizer(
          ngram_range=(1, 2),
          max_features=50000,
          lowercase=True,
          strip_accents="ascii",
          min_df=2
      )),
      ("clf", LinearSVC())
  ])

  clf_pipeline.fit(train_df["combined_text"], train_df["label_id"])

  val_preds = clf_pipeline.predict(val_df["combined_text"])

  print("Validation classification report:")
  print(classification_report(
      val_df["label_id"],
      val_preds,
      target_names=[id2label[i] for i in sorted(id2label.keys())]
  ))

  print("Confusion matrix (val):")
  print(confusion_matrix(val_df["label_id"], val_preds))

  if save_path is not None:
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    joblib.dump(clf_pipeline, save_path)
    print(f"Saved baseline model to {save_path}")

  return clf_pipeline

In [ ]:
baseline_model = train_baseline_tfidf_model(
    train_df,
    val_df,
    ID2LABEL,
    save_path="models/baseline_tfidf_linearsvc.joblib"
)

In [ ]:
def evaluate_on_test(model, test_df, id2label):
    test_preds = model.predict(test_df["combined_text"])
    print("Test classification report:")
    print(classification_report(
        test_df["label_id"],
        test_preds,
        target_names=[id2label[i] for i in sorted(id2label.keys())]
    ))
    print("Confusion matrix (test):")
    print(confusion_matrix(test_df["label_id"], test_preds))

evaluate_on_test(baseline_model, test_df, ID2LABEL)

# TF-IDF + LinearSVC (with balanced class weight)

In [ ]:
from sklearn.svm import LinearSVC

clf_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=50000,
        lowercase=True,
        strip_accents="unicode",
        min_df=2
    )),
    ("clf", LinearSVC(class_weight="balanced"))
])

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)
import numpy as np

def evaluate_on_test(model, test_df, id2label):
    y_true = test_df["label_id"].values
    y_pred = model.predict(test_df["combined_text"])

    print("Test classification report:")
    print(classification_report(
        y_true,
        y_pred,
        target_names=[id2label[i] for i in sorted(id2label.keys())]
    ))

    cm = confusion_matrix(y_true, y_pred, labels=sorted(id2label.keys()))
    print("Confusion matrix (rows = true, cols = pred, label order =", sorted(id2label.keys()), ")")
    print(cm)

    si_label_id = 3
    si_index = sorted(id2label.keys()).index(si_label_id)

    si_true = (y_true == si_label_id)
    si_pred = (y_pred == si_label_id)

    si_tp = np.sum(si_true & si_pred)
    si_fn = np.sum(si_true & ~si_pred)
    si_fp = np.sum(~si_true & si_pred)

    si_recall = si_tp / (si_tp + si_fn) if (si_tp + si_fn) > 0 else 0.0
    si_precision = si_tp / (si_tp + si_fp) if (si_tp + si_fp) > 0 else 0.0

    print(f"\nSI analysis (label_id={si_label_id}):")
    print(f"  TP: {si_tp}, FN: {si_fn}, FP: {si_fp}")
    print(f"  SI recall:    {si_recall:.3f}")
    print(f"  SI precision: {si_precision:.3f}")

evaluate_on_test(baseline_model, test_df, ID2LABEL)

# **LIME + LinearSVC**

In [ ]:
!pip install lime

In [ ]:
from lime.lime_text import LimeTextExplainer
import numpy as np

def make_predict_proba_fn(pipeline, n_classes):
  def predict_proba(texts):
    decision = pipeline.decision_function(texts)

    exp_scores = np.exp(decision - np.max(decision, axis=1, keepdims=True))
    probs = exp_scores / exp_scores.sum(axis=1, keepdims=True)
    return probs
  return predict_proba

class_names = [ID2LABEL[i] for i in sorted(ID2LABEL.keys())]
predict_proba_fn = make_predict_proba_fn(baseline_model, len(class_names))
explainer = LimeTextExplainer(class_names=class_names)

In [ ]:
idx = 6
text_to_explain = test_df["combined_text"].iloc[idx]
true_label = ID2LABEL[test_df["label_id"].iloc[idx]]

print("TEXT")
print(text_to_explain)
print("\nTrue label:", true_label)

exp = explainer.explain_instance(
    text_to_explain,
    predict_proba_fn,
    num_features=10,
    top_labels=1
)

pred_label_idx = exp.top_labels[0]
print("\nPredicted label:", class_names[pred_label_idx])
exp.show_in_notebook(text=text_to_explain)

# **Weighted-loss DistilBERT 4-class model with max_length = 256**

In [ ]:
# !pip install transformers --upgrade --force-reinstall
!pip install transformers datasets accelerate -q

In [ ]:
import torch
from torch import nn
from datasets import Dataset, DatasetDict
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

LABEL2ID = {"NEU": 0, "HUMOR": 1, "MH": 2, "SI": 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
num_labels = len(LABEL2ID)

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def df_to_hf_dataset(df):
  return Dataset.from_pandas(
      df[["combined_text", "label_id"]].rename(columns={"label_id": "labels"})
  )

raw_datasets = DatasetDict({
    "train": df_to_hf_dataset(train_df),
    "validation": df_to_hf_dataset(val_df),
    "test": df_to_hf_dataset(test_df),
})

def tokenizer_batch(batch):
  return tokenizer(
      batch["combined_text"],
      padding="max_length",
      truncation=True,
      max_length=256
  )

tokenized_datasets = raw_datasets.map(
    tokenizer_batch,
    batched=True,
)

tokenized_datasets = tokenized_datasets.remove_columns(["combined_text"])
tokenized_datasets.set_format("torch")

train_labels = train_df["label_id"].values
class_counts = np.bincount(train_labels, minlength=num_labels)
class_counts

inv_freq = 1.0 / class_counts
class_weights = inv_freq / inv_freq.sum() * num_labels

print("Class counts:", class_counts)
print("Class wieghts:", class_weights)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

class WeightedTrainer(Trainer):
  def __init__(self, class_weights, *args, **kwargs):
    _ = kwargs.pop('tokenizer', None)
    super().__init__(*args, **kwargs)
    self.class_weights = class_weights.to(self.model.device)

  def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
    labels = inputs.get("labels")
    outputs = model(**inputs)
    logits = outputs.get("logits")
    loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
    loss = loss_fct(logits.view(-1, model.num_labels), labels.view(-1))
    return (loss, outputs) if return_outputs else loss

In [ ]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)
  report = classification_report(
      labels,
      predictions,
      target_names=[ID2LABEL[i] for i in range(num_labels)],
      output_dict=True
  )

  macro_f1 = report["macro avg"]["f1-score"]
  accuracy = report["accuracy"]

  si_idx = LABEL2ID["SI"]
  si_recall = report[ID2LABEL[si_idx]]["recall"]

  return {
      "accuracy": accuracy,
      "macro_f1": macro_f1,
      "si_recall": si_recall,
  }

In [ ]:
batch_size = 16

training_args = TrainingArguments(
    output_dir="./distilbert-4class",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    optim="adamw_torch",
)

trainer = WeightedTrainer(
    class_weights=class_weights_tensor,
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

test_results = trainer.predict(tokenized_datasets["test"])

In [ ]:
logits = test_results.predictions
labels = test_results.label_ids
preds = np.argmax(logits, axis=-1)

print("Test classification report (DistilBERT):")
print(classification_report(
    labels,
    preds,
    target_names=[ID2LABEL[i] for i in range(num_labels)]
))
print("Confusion matrix (test):")
print(confusion_matrix(labels, preds))

In [ ]:
output_dir = "./models/distilbert-4class-model"
model.cpu().save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model and tokenizer saved to {output_dir}")

### Prediction Helper Function

In [ ]:
def predict_entry(text):
  inputs = tokenizer(
      text,
      padding='max_length',
      truncation=True,
      max_length=256,
      return_tensors='pt'
  )
  inputs = {k: v.to(model.device) for k, v in inputs.items()}

  with torch.no_grad():
    outputs = model(**inputs)
  logits = outputs.logits

  probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

  predicted_label_id = np.argmax(probs)
  predicted_label = ID2LABEL[predicted_label_id]

  probs_dict = {ID2LABEL[i]: prob for i, prob in enumerate(probs)}

  return predicted_label, probs_dict

example_text = "I am feeling very sad and don't know what to do anymore."
predicted_label, probabilities = predict_entry(example_text)

print(f"Example Text: {example_text}")
print(f"Predicted Label: {predicted_label}")
print("Probabilities:")
for label, prob in probabilities.items():
  print(f"  {label}: {prob:.4f}")

example_text_2 = "This joke is absolutely hilarious, I can't stop laughing!"
predicted_label_2, probabilities_2 = predict_entry(example_text_2)

print(f"\nExample Text 2: {example_text_2}")
print(f"Predicted Label: {predicted_label_2}")
print("Probabilities:")
for label, prob in probabilities_2.items():
  print(f"  {label}: {prob:.4f}")

In [ ]:
import shutil
import os

output_zip_name = 'models.zip'
shutil.make_archive(os.path.splitext(output_zip_name)[0], 'zip', './models')
print(f"Folder 'models' compressed to {output_zip_name}. You can download it from the Colab file browser (left sidebar -> Files tab).")

# **LIME + DistilBERT 4-class model**

In [ ]:
import torch
import numpy as np
from lime.lime_text import LimeTextExplainer
import torch.nn.functional as F

class_names = [ID2LABEL[i] for i in range(len(ID2LABEL))]

def make_hf_predict_proba_fn(model, tokenizer, max_length=256, device=None):
  if device is None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  model.to(device)
  model.eval()

  def predict_proba(texts):
    encodings = tokenizer(
       texts,
       padding=True,
       max_length=max_length,
       return_tensors="pt"
    )
    encodings = {k: v.to(device) for k, v in encodings.items()}

    with torch.no_grad():
      outputs = model(**encodings)
      logits = outputs.logits
      probs = F.softmax(logits, dim=-1).cpu().numpy()
    return probs

  return predict_proba

predict_proba_fn = make_hf_predict_proba_fn(model, tokenizer, max_length=256)
explainer = LimeTextExplainer(class_names=class_names)

In [ ]:
idx = 897
text_to_explain = test_df["combined_text"].iloc[idx]
true_label = ID2LABEL[test_df["label_id"].iloc[idx]]

print("Text:\n", text_to_explain)
print("\nTrue label:", true_label)

exp = explainer.explain_instance(
    text_to_explain,
    predict_proba_fn,
    num_features=10,
    top_labels=1
)

pred_label_idx = exp.top_labels[0]
print("\nPredicted label:", class_names[pred_label_idx])

exp.show_in_notebook(text=text_to_explain)

# **Decision Logic (DistilBERT outputs with PHQ/GAD scores)**

In [ ]:
def score_phq9(items):

  assert len(items) == 9
  total = sum(items)

  if total <= 4:
    severity = "minimal"
  elif total <= 9:
    serverity = "mild"
  elif total <= 14:
    severity = "moderate"
  elif total <= 19:
    severity = "moderately severe"
  else:
    severity = "severe"

  return total, severity

def score_phq2(items):

  assert len(items) == 2
  total = sum(items)

  flag_full = total >= 3
  return total, flag_full

def score_gad7(items):

  assert len(items) == 7
  total = sum(items)

  if total <= 4:
    severity = "minimal"
  elif total <= 9:
    severity = "mild"
  elif total <= 14:
    severity = "moderate"
  else:
    severity = "severe"

  return total, severity

def score_gad2(items):

  assert len(items) == 2
  total = sum(items)
  flag_full = total >= 3
  return total, flag_full

def classify_risk_from_model(pred_label, probs, si_label="SI", mh_label="MH"):

  si_prob = probs.get(si_label, 0.0)
  mh_prob = probs.get(mh_label, 0.0)

  if pred_label == si_label or si_prob >= 0.5:
    return "high"
  elif pred_label == mh_label or mh_prob >= 0.5 or si_prob >= 0.3:
    return "elevated"
  else:
    return "low"

In [ ]:
def decide_next_step(
    model_label,
    model_probs,
    phq9_items=None,
    gad7_items=None,
):

  risk_tier = classify_risk_from_model(model_label, model_probs)

  result = {
      "risk_tier": risk_tier,
      "show_crisis_card": False,
      "show_phq_card": False,
      "recomendation": "genernal_positive_feedback",
      "phq_summary": None,
      "gad_summary": None,
      "sucicidality_flag": False,
  }

  if risk_tier == "high":
    result["show_crisis_card"] = True
    result["recommendation"] = "crisis_resources"

  if risk_tier in ["elevated", "high"]:
    result["show_phq_gad_prompt"] = True

  if phq9_items is not None:
    phq_total, phq_severity = score_phq9(phq9_items)
    phq_item9 = phq9_items[8]
    suicidality_flag = phq9_items > 0
    result["phq_summary"] = {
        "total": phq_total,
        "severity": phq_severity,
        "item9_positive": bool(suicidality_flag)
    }
    result["suicidality_flag"] = bool(suicidality_flag)

    if suicidality_flag:
      result["show_crisis_card"] = True
      result["recommendation"] = "crisis_resources"
    elif phq_total >= 15:
      result["recommendation"] = "seek_professional_help"
    elif phq_total >= 10:
      result["recommendation"] = "consider_professional_help"
    elif phq_total >= 5:
      result["recommendation"] = "monitor_and_self_care"


  if gad7_items is not None:
    gad_total, gad_severity = score_gad7(gad7_items)
    result["gad_summary"] = {
        "total": gad_total,
        "severity": gad_severity,
    }
    if gad_total >= 15:
      result["recommnedation"] = "seek_professional_help"
    elif gad_total >= 10 and result["recommendation"] not in ["seek_rpofessional_help", "crisis_resources"]:
      result["recommendation"] = "consider_professional_help"

  return result